# TransformerLab AI
### Interactive NLP Intelligence Workbench

A Transformer-powered NLP workbench built using Hugging Face Transformers.

Features:
- Sentiment Analysis
- Zero-Shot Classification
- Named Entity Recognition
- Question Answering
- Summarization
- Translation
- Text Generation
- Semantic Search
- Model Insights
- Model Comparison

In [1]:
!pip install -q transformers sentence-transformers gradio pypdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.0 MB/s eta 0:00:00


In [2]:
import torch
import transformers
import gradio as gr

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)

device = 0 if torch.cuda.is_available() else -1

if torch.cuda.is_available():
    print(" GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.11.0+cpu
Transformers: 5.17.0
Gradio: 6.27.0
Running on CPU


In [3]:
from transformers import pipeline

In [4]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [12]:
text = ["I absolutely loved learning about Transformer models!","i hate thisss so much "]

result = sentiment_model(text)

result

[{'label': 'POSITIVE', 'score': 0.99982750415802},
 {'label': 'NEGATIVE', 'score': 0.9992965459823608}]

In [13]:
def analyze_sentiment(text):

    if not text.strip():
        return {
            "error": "Please enter some text."
        }

    result = sentiment_model(text)[0]

    return {
        "task": "Sentiment Analysis",
        "label": result["label"],
        "confidence": round(result["score"] * 100, 2),
        "model": "DistilBERT SST-2"
    }

In [15]:
tokenizer = sentiment_model.tokenizer

sample_text = "Transformers are incredibly powerful!"

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original:")
print(sample_text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

Original:
Transformers are incredibly powerful!

Tokens:
['transformers', 'are', 'incredibly', 'powerful', '!']

Token IDs:
[19081, 2024, 11757, 3928, 999]


In [16]:
def sentiment_ui(text):

    result = analyze_sentiment(text)

    if "error" in result:
        return result["error"]

    return (
        f"Sentiment: {result['label']}\n"
        f"Confidence: {result['confidence']}%\n"
        f"Model: {result['model']}"
    )


demo = gr.Interface(
    fn=sentiment_ui,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Enter text to analyze..."
    ),
    outputs=gr.Textbox(label="Analysis"),
    title="⚡ TransformerLab AI",
    description="Transformer-powered NLP Intelligence Workbench"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a65049deabcf8dafc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Module 2 — Zero-Shot Classification

Classify text into user-defined categories without task-specific training.

In [21]:
zero_shot_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [22]:
text = """
Researchers developed a new artificial intelligence system
that can detect diseases from medical images.
"""

labels = [
    "artificial intelligence",
    "healthcare",
    "finance",
    "sports",
    "politics"
]

result = zero_shot_model(
    text,
    candidate_labels=labels
)

result

{'sequence': '\nResearchers developed a new artificial intelligence system\nthat can detect diseases from medical images.\n',
 'labels': ['artificial intelligence',
  'healthcare',
  'politics',
  'finance',
  'sports'],
 'scores': [0.8650214076042175,
  0.10803008824586868,
  0.01025998592376709,
  0.00897091906517744,
  0.007717554923146963]}

In [24]:
def classify_zero_shot(text, labels):

    if not text.strip():
        return {"error": "Please enter some text."}

    if not labels:
        return {"error": "Please provide candidate labels."}

    result = zero_shot_model(
        text,
        candidate_labels=labels
    )

    predictions = []

    for label, score in zip(
        result["labels"],
        result["scores"]
    ):
        predictions.append({
            "label": label,
            "confidence": round(score * 100, 2)
        })

    return {
        "task": "Zero-Shot Classification",
        "predictions": predictions,
        "model": "BART Large MNLI"
    }

In [25]:
def zero_shot_ui(text, labels):

    label_list = [
        label.strip()
        for label in labels.split(",")
        if label.strip()
    ]

    result = classify_zero_shot(
        text,
        label_list
    )

    if "error" in result:
        return result["error"]

    output = ""

    for prediction in result["predictions"]:
        output += (
            f"{prediction['label']}: "
            f"{prediction['confidence']}%\n"
        )

    return output


zero_demo = gr.Interface(
    fn=zero_shot_ui,

    inputs=[
        gr.Textbox(
            lines=5,
            label="Text",
            placeholder="Enter text..."
        ),

        gr.Textbox(
            label="Candidate Labels",
            placeholder="technology, sports, politics, business"
        )
    ],

    outputs=gr.Textbox(
        label="Classification"
    ),

    title="🏷️ Zero-Shot Classification"
)

zero_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f47b96ef3fd17e5221.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Module 3 — Named Entity Recognition (NER)

Identify people, organizations, locations, and other named entities from text.

In [30]:
def ner_ui(text):

    result = analyze_entities(text)

    if "error" in result:
        return result["error"]

    if not result["entities"]:
        return "No named entities detected."

    output = ""

    for entity in result["entities"]:

        output += (
            f"{entity['text']} → "
            f"{entity['type']} "
            f"({entity['confidence']}%)\n"
        )

    return output

In [31]:
analyze_entities(
    """
    Sundar Pichai announced that Google will invest
    in artificial intelligence research in London.
    """
)

{'task': 'Named Entity Recognition',
 'entities': [{'text': 'Sundar Pichai',
   'type': 'PER',
   'confidence': 98.95,
   'start': 5,
   'end': 18},
  {'text': 'Google',
   'type': 'ORG',
   'confidence': 99.84,
   'start': 34,
   'end': 40},
  {'text': 'London',
   'type': 'LOC',
   'confidence': 99.95,
   'start': 96,
   'end': 102}],
 'model': 'BERT Base NER'}

In [32]:
ner_demo = gr.Interface(
    fn=ner_ui,

    inputs=gr.Textbox(
        lines=7,
        label="Text",
        placeholder="Enter text containing people, companies, locations..."
    ),

    outputs=gr.Textbox(
        label="Detected Entities"
    ),

    title="👤 Named Entity Recognition",
    description="Detect people, organizations and locations using BERT."
)

ner_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a1217b211c87ab665.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


##Module 4 — Extractive Question Answering

Ask natural-language questions about a supplied context and extract the answer directly from that context.

In [50]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Move model to device
if device == 0 and torch.cuda.is_available():
    model = model.to("cuda")

def qa_model(question, context):
    inputs = tokenizer(question, context, return_offsets_mapping=True, return_tensors="pt")
    offset_mapping = inputs.pop("offset_mapping")[0]

    if device == 0 and torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Get start and end logits
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Compute probabilities
    start_probs = torch.softmax(start_logits, dim=-1)
    end_probs = torch.softmax(end_logits, dim=-1)

    # Find best positions
    answer_start = torch.argmax(start_logits).item()
    answer_end = torch.argmax(end_logits).item()

    # Calculate score/confidence
    score = (start_probs[answer_start] * end_probs[answer_end]).item()

    # Decode the answer text
    answer = tokenizer.convert_tokens_to_string(
        tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start:answer_end + 1])
    )

    # Determine character-level start and end in original context
    sequence_ids = inputs.sequence_ids(0)
    char_start = 0
    char_end = 0

    # Filter offsets corresponding only to context tokens
    for idx, (start, end) in enumerate(offset_mapping):
        if sequence_ids[idx] == 1:  # Context tokens have sequence_id of 1
            if idx == answer_start:
                char_start = start.item()
            if idx == answer_end:
                char_end = end.item()
                break

    if char_end == 0:
         char_end = char_start + len(answer.strip())

    return {
        "answer": answer,
        "score": score,
        "start": char_start,
        "end": char_end
    }

print("✅ Rebuilt QA model with full support for score, start, and end tracking!")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✅ Rebuilt QA model with full support for score, start, and end tracking!


In [53]:
context = """
Microsoft announced that it will invest $3 billion to expand
its artificial intelligence and cloud infrastructure in India.
The investment will take place over the next two years and
will include new data centers and AI training initiatives.
"""

question = "How much will Microsoft invest?"

result = qa_model(
    question=question,
    context=context
)

result

{'answer': '$ 3 billion', 'score': 0.9894534349441528, 'start': 41, 'end': 51}

In [54]:
qa_model(
    question="Where will Microsoft expand its infrastructure?",
    context=context
)

{'answer': 'India', 'score': 0.935343861579895, 'start': 118, 'end': 123}

In [55]:
def answer_question(question, context):

    if not question.strip():
        return {
            "error": "Please enter a question."
        }

    if not context.strip():
        return {
            "error": "Please provide some context."
        }

    result = qa_model(
        question=question,
        context=context
    )

    return {
        "task": "Extractive Question Answering",
        "answer": result["answer"],
        "confidence": round(
            float(result["score"]) * 100,
            2
        ),
        "start": result["start"],
        "end": result["end"],
        "model": "DistilBERT SQuAD"
    }

In [56]:
def highlight_answer(context, start, end):

    before = context[:start]
    answer = context[start:end]
    after = context[end:]

    return before + "**" + answer + "**" + after

In [64]:
result = answer_question(
    "How much will Microsoft invest?",
    context
)

highlight_answer(
    context,
    result["start"],
    result["end"]
)

'\nMicrosoft announced that it will invest **$3 billion** to expand\nits artificial intelligence and cloud infrastructure in India.\nThe investment will take place over the next two years and\nwill include new data centers and AI training initiatives.\n'

In [60]:
def qa_ui(context, question):

    result = answer_question(
        question,
        context
    )

    if "error" in result:
        return result["error"], ""

    answer_output = (
        f"Answer: {result['answer']}\n\n"
        f"Confidence: {result['confidence']}%\n\n"
        f"Model: {result['model']}"
    )

    evidence = highlight_answer(
        context,
        result["start"],
        result["end"]
    )

    return answer_output, evidence

In [61]:
qa_demo = gr.Interface(
    fn=qa_ui,

    inputs=[
        gr.Textbox(
            lines=10,
            label="Context",
            placeholder="Paste the information here..."
        ),

        gr.Textbox(
            label="Question",
            placeholder="Ask a question about the context..."
        )
    ],

    outputs=[
        gr.Textbox(
            label="Answer"
        ),

        gr.Markdown(
            label="Source Evidence"
        )
    ],

    title="❓ Transformer Question Answering",
    description="Ask questions and extract answers directly from the supplied context."
)

qa_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4928213efe0c592c49.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [65]:
# ============================================================
# MODULE 5 — TEXT SUMMARIZATION
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarization_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to device
if device == 0 and torch.cuda.is_available():
    summarization_model = summarization_model.to("cuda")

def summarize_text(text, length="Medium"):

    if not text or not text.strip():
        return {
            "error": "Please enter some text."
        }

    word_count = len(text.split())

    # Prevent very short text from being summarized
    if word_count < 40:
        return {
            "error": "Please provide at least 40 words for summarization."
        }

    # Summary length settings
    length_settings = {
        "Short": {
            "max_length": 60,
            "min_length": 20
        },
        "Medium": {
            "max_length": 120,
            "min_length": 40
        },
        "Detailed": {
            "max_length": 180,
            "min_length": 70
        }
    }

    settings = length_settings.get(
        length,
        length_settings["Medium"]
    )

    # Tokenize input text
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")
    if device == 0 and torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate summary tokens
    with torch.no_grad():
        summary_ids = summarization_model.generate(
            inputs["input_ids"],
            max_length=settings["max_length"],
            min_length=settings["min_length"],
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    original_words = len(text.split())
    summary_words = len(summary.split())

    compression = round(
        (1 - (summary_words / original_words)) * 100,
        2
    )

    return {
        "task": "Text Summarization",
        "summary": summary,
        "original_words": original_words,
        "summary_words": summary_words,
        "compression": compression,
        "model": "DistilBART CNN"
    }

def summarization_ui(text, length):

    result = summarize_text(text, length)

    if "error" in result:
        return result["error"]

    return f"""
SUMMARY

{result['summary']}

----------------------------------

Original Length: {result['original_words']} words
Summary Length: {result['summary_words']} words
Compression: {result['compression']}%

Model: {result['model']}
"""

summarization_demo = gr.Interface(
    fn=summarization_ui,
    inputs=[
        gr.Textbox(
            lines=15,
            label="Text",
            placeholder="Paste a long article or document here..."
        ),
        gr.Radio(
            choices=[
                "Short",
                "Medium",
                "Detailed"
            ],
            value="Medium",
            label="Summary Length"
        )
    ],
    outputs=gr.Textbox(
        lines=12,
        label="Summary"
    ),
    title="📝 TransformerLab — Text Summarization",
    description=(
        "Generate concise summaries using a "
        "Transformer-based sequence-to-sequence model."
    )
)

summarization_demo.launch()

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e034e59752c6496b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [67]:
# ============================================================
# MODULE 6 — MULTILINGUAL TRANSLATION
# TransformerLab AI
# Pipeline-independent version
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# ------------------------------------------------------------
# 1. MODEL CONFIGURATION
# ------------------------------------------------------------

TRANSLATION_MODELS = {

    "English → French":
        "Helsinki-NLP/opus-mt-en-fr",

    "English → German":
        "Helsinki-NLP/opus-mt-en-de",

    "English → Hindi":
        "Helsinki-NLP/opus-mt-en-hi",

    "French → English":
        "Helsinki-NLP/opus-mt-fr-en",

    "German → English":
        "Helsinki-NLP/opus-mt-de-en"
}


# ------------------------------------------------------------
# 2. MODEL CACHE
# ------------------------------------------------------------

translation_cache = {}


def get_translation_model(language_pair):

    if language_pair not in TRANSLATION_MODELS:

        raise ValueError(
            f"Unsupported language pair: {language_pair}"
        )

    # Only load model when first requested
    if language_pair not in translation_cache:

        model_name = TRANSLATION_MODELS[language_pair]

        print(f"Loading: {model_name}")

        tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )

        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name
        )

        # Move to GPU if available
        if torch.cuda.is_available():
            model = model.to("cuda")

        model.eval()

        translation_cache[language_pair] = {
            "tokenizer": tokenizer,
            "model": model
        }

    return translation_cache[language_pair]


# ------------------------------------------------------------
# 3. TRANSLATION FUNCTION
# ------------------------------------------------------------

def translate_text(text, language_pair):

    if not text or not text.strip():

        return {
            "error": "Please enter text to translate."
        }

    try:

        translator = get_translation_model(
            language_pair
        )

        tokenizer = translator["tokenizer"]
        model = translator["model"]

        # -----------------------------------------
        # TOKENIZATION
        # -----------------------------------------

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Move input tensors to same device as model
        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

        # -----------------------------------------
        # GENERATION
        # -----------------------------------------

        with torch.no_grad():

            generated_tokens = model.generate(
                **inputs,
                max_new_tokens=256,
                num_beams=4,
                early_stopping=True
            )

        # -----------------------------------------
        # DECODING
        # -----------------------------------------

        translated_text = tokenizer.decode(
            generated_tokens[0],
            skip_special_tokens=True
        )

        return {

            "task":
                "Machine Translation",

            "translation":
                translated_text,

            "language_pair":
                language_pair,

            "model":
                TRANSLATION_MODELS[language_pair],

            "input_tokens":
                int(inputs["input_ids"].shape[1]),

            "output_tokens":
                int(generated_tokens.shape[1])
        }

    except Exception as e:

        return {
            "error": str(e)
        }


# ------------------------------------------------------------
# 4. GRADIO FUNCTION
# ------------------------------------------------------------

def translation_ui(text, language_pair):

    result = translate_text(
        text,
        language_pair
    )

    if "error" in result:

        return (
            f"Translation error:\n{result['error']}",
            ""
        )

    model_info = f"""
Language Pair: {result['language_pair']}

Model: {result['model']}

Input Tokens: {result['input_tokens']}

Output Tokens: {result['output_tokens']}
"""

    return (
        result["translation"],
        model_info
    )


# ------------------------------------------------------------
# 5. GRADIO INTERFACE
# ------------------------------------------------------------

translation_demo = gr.Interface(

    fn=translation_ui,

    inputs=[

        gr.Textbox(
            lines=8,
            label="Text",
            placeholder="Enter text to translate..."
        ),

        gr.Dropdown(
            choices=list(
                TRANSLATION_MODELS.keys()
            ),
            value="English → French",
            label="Language Pair"
        )
    ],

    outputs=[

        gr.Textbox(
            lines=8,
            label="Translation"
        ),

        gr.Textbox(
            lines=8,
            label="Model Information"
        )
    ],

    title="🌍 TransformerLab — Neural Machine Translation",

    description=(
        "Translate text using Transformer-based "
        "sequence-to-sequence models."
    )
)


# ------------------------------------------------------------
# 6. LAUNCH
# ------------------------------------------------------------

translation_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1327e8e8e944b993de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [68]:
# ============================================================
# MODULE 7 — TRANSFORMER TEXT GENERATION
# TransformerLab AI
# ============================================================

import time
from transformers import pipeline


# ------------------------------------------------------------
# 1. LOAD MODEL
# ------------------------------------------------------------

GENERATION_MODEL = "HuggingFaceTB/SmolLM2-360M"

text_generator = pipeline(
    "text-generation",
    model=GENERATION_MODEL,
    device=device
)


# ------------------------------------------------------------
# 2. GENERATION FUNCTION
# ------------------------------------------------------------

def generate_text(
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9,
    num_sequences=1
):

    if not prompt or not prompt.strip():
        return {
            "error": "Please enter a prompt."
        }

    temperature = float(temperature)
    top_p = float(top_p)
    max_new_tokens = int(max_new_tokens)
    num_sequences = int(num_sequences)

    start_time = time.time()

    results = text_generator(
        prompt,

        max_new_tokens=max_new_tokens,

        do_sample=True,

        temperature=temperature,

        top_p=top_p,

        num_return_sequences=num_sequences,

        pad_token_id=text_generator.tokenizer.eos_token_id
    )

    inference_time = (
        time.time() - start_time
    ) * 1000

    generations = []

    for index, result in enumerate(results):

        full_text = result["generated_text"]

        # Remove the original prompt from output
        if full_text.startswith(prompt):

            generated_part = full_text[
                len(prompt):
            ].strip()

        else:

            generated_part = full_text.strip()

        generations.append({
            "sequence": index + 1,
            "text": generated_part
        })

    # Token information
    input_tokens = text_generator.tokenizer.encode(
        prompt
    )

    return {

        "task":
            "Text Generation",

        "generations":
            generations,

        "model":
            GENERATION_MODEL,

        "temperature":
            temperature,

        "top_p":
            top_p,

        "max_new_tokens":
            max_new_tokens,

        "input_tokens":
            len(input_tokens),

        "inference_time_ms":
            round(inference_time, 2)
    }


# ------------------------------------------------------------
# 3. GRADIO FUNCTION
# ------------------------------------------------------------

def generation_ui(
    prompt,
    max_new_tokens,
    temperature,
    top_p,
    num_sequences
):

    try:

        result = generate_text(
            prompt,
            max_new_tokens,
            temperature,
            top_p,
            num_sequences
        )

        if "error" in result:

            return (
                result["error"],
                ""
            )

        output = ""

        for generation in result["generations"]:

            output += (
                f"GENERATION "
                f"{generation['sequence']}\n\n"
                f"{generation['text']}\n\n"
                f"{'-' * 50}\n\n"
            )

        model_info = f"""
Model: {result['model']}

Input Tokens: {result['input_tokens']}

Maximum New Tokens: {result['max_new_tokens']}

Temperature: {result['temperature']}

Top-p: {result['top_p']}

Inference Time: {result['inference_time_ms']} ms
"""

        return (
            output,
            model_info
        )

    except Exception as e:

        return (
            f"Generation error:\n{str(e)}",
            ""
        )


# ------------------------------------------------------------
# 4. GRADIO INTERFACE
# ------------------------------------------------------------

generation_demo = gr.Interface(

    fn=generation_ui,

    inputs=[

        gr.Textbox(
            lines=6,
            label="Prompt",
            placeholder="Enter a prompt..."
        ),

        gr.Slider(
            minimum=20,
            maximum=250,
            value=100,
            step=10,
            label="Maximum New Tokens"
        ),

        gr.Slider(
            minimum=0.1,
            maximum=1.5,
            value=0.7,
            step=0.1,
            label="Temperature"
        ),

        gr.Slider(
            minimum=0.1,
            maximum=1.0,
            value=0.9,
            step=0.05,
            label="Top-p"
        ),

        gr.Slider(
            minimum=1,
            maximum=3,
            value=1,
            step=1,
            label="Number of Generations"
        )
    ],

    outputs=[

        gr.Textbox(
            lines=15,
            label="Generated Text"
        ),

        gr.Textbox(
            lines=8,
            label="Model Information"
        )
    ],

    title="✨ TransformerLab — Text Generation",

    description=(
        "Experiment with autoregressive Transformer "
        "text generation and decoding parameters."
    )
)


# ------------------------------------------------------------
# 5. LAUNCH
# ------------------------------------------------------------

generation_demo.launch()

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ea17ce213098a7b727.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [69]:
# ============================================================
# MODULE 8 — SEMANTIC SEARCH
# TransformerLab AI
# ============================================================

import time
import numpy as np
import gradio as gr

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. LOAD EMBEDDING MODEL
# ------------------------------------------------------------

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


# ------------------------------------------------------------
# 2. TEXT CHUNKING
# ------------------------------------------------------------

def chunk_text(text, chunk_size=80):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        if chunk.strip():
            chunks.append(chunk)

    return chunks


# ------------------------------------------------------------
# 3. SEMANTIC SEARCH FUNCTION
# ------------------------------------------------------------

def semantic_search(
    document,
    query,
    top_k=3
):

    if not document or not document.strip():
        return {
            "error": "Please provide a document."
        }

    if not query or not query.strip():
        return {
            "error": "Please enter a search query."
        }

    start_time = time.time()


    # --------------------------------------------------------
    # STEP 1 — CHUNK DOCUMENT
    # --------------------------------------------------------

    chunks = chunk_text(document)


    # --------------------------------------------------------
    # STEP 2 — CREATE CHUNK EMBEDDINGS
    # --------------------------------------------------------

    chunk_embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # STEP 3 — CREATE QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # STEP 4 — COSINE SIMILARITY
    # --------------------------------------------------------

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]


    # --------------------------------------------------------
    # STEP 5 — RANK RESULTS
    # --------------------------------------------------------

    top_k = min(
        int(top_k),
        len(chunks)
    )

    ranked_indices = np.argsort(
        similarities
    )[::-1][:top_k]


    results = []

    for rank, index in enumerate(
        ranked_indices,
        start=1
    ):

        results.append({

            "rank":
                rank,

            "chunk":
                chunks[index],

            "similarity":
                round(
                    float(similarities[index]),
                    4
                )
        })


    search_time = round(
        (time.time() - start_time) * 1000,
        2
    )


    return {

        "task":
            "Semantic Search",

        "query":
            query,

        "results":
            results,

        "total_chunks":
            len(chunks),

        "embedding_dimension":
            int(chunk_embeddings.shape[1]),

        "search_time_ms":
            search_time,

        "model":
            EMBEDDING_MODEL
    }


# ------------------------------------------------------------
# 4. GRADIO FUNCTION
# ------------------------------------------------------------

def semantic_search_ui(
    document,
    query,
    top_k
):

    try:

        result = semantic_search(
            document,
            query,
            top_k
        )

        if "error" in result:

            return (
                result["error"],
                ""
            )


        output = ""

        for item in result["results"]:

            output += (
                f"RANK {item['rank']}\n"
                f"Similarity: "
                f"{item['similarity']}\n\n"
                f"{item['chunk']}\n\n"
                f"{'-' * 60}\n\n"
            )


        model_info = f"""
Model: {result['model']}

Document Chunks: {result['total_chunks']}

Embedding Dimension: {result['embedding_dimension']}

Search Time: {result['search_time_ms']} ms
"""


        return (
            output,
            model_info
        )


    except Exception as e:

        return (
            f"Semantic search error:\n{str(e)}",
            ""
        )


# ------------------------------------------------------------
# 5. GRADIO INTERFACE
# ------------------------------------------------------------

semantic_demo = gr.Interface(

    fn=semantic_search_ui,

    inputs=[

        gr.Textbox(
            lines=15,
            label="Document",
            placeholder="Paste a document here..."
        ),

        gr.Textbox(
            label="Search Query",
            placeholder="What information are you looking for?"
        ),

        gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Top Results"
        )
    ],

    outputs=[

        gr.Textbox(
            lines=15,
            label="Semantic Search Results"
        ),

        gr.Textbox(
            lines=6,
            label="Embedding Information"
        )
    ],

    title="🔎 TransformerLab — Semantic Search",

    description=(
        "Search documents by meaning using "
        "Transformer embeddings and cosine similarity."
    )
)


# ------------------------------------------------------------
# 6. LAUNCH
# ------------------------------------------------------------

semantic_demo.launch()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ef6c9c5a4021c8774.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [70]:
# ============================================================
# MODULE 9 — PDF/TXT DOCUMENT INTELLIGENCE
# TransformerLab AI
# ============================================================

import os
import re
import time
import numpy as np
import gradio as gr

from pypdf import PdfReader
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. TEXT CLEANING
# ------------------------------------------------------------

def clean_document_text(text):

    if not text:
        return ""

    # Remove excessive spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce excessive newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------------------------
# 2. PDF EXTRACTION
# ------------------------------------------------------------

def extract_pdf_text(file_path):

    reader = PdfReader(file_path)

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):

        page_text = page.extract_text()

        if page_text:

            page_text = clean_document_text(
                page_text
            )

            pages.append({
                "page": page_number,
                "text": page_text
            })

    return pages


# ------------------------------------------------------------
# 3. TXT EXTRACTION
# ------------------------------------------------------------

def extract_txt_text(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        text = file.read()

    text = clean_document_text(text)

    return [{
        "page": 1,
        "text": text
    }]


# ------------------------------------------------------------
# 4. DOCUMENT LOADER
# ------------------------------------------------------------

def load_document(file_path):

    if not file_path:

        return {
            "error": "Please upload a document."
        }

    extension = os.path.splitext(
        file_path
    )[1].lower()

    if extension == ".pdf":

        pages = extract_pdf_text(
            file_path
        )

        file_type = "PDF"

    elif extension == ".txt":

        pages = extract_txt_text(
            file_path
        )

        file_type = "TXT"

    else:

        return {
            "error":
                "Unsupported file type. "
                "Please upload PDF or TXT."
        }


    if not pages:

        return {
            "error":
                "No readable text was found."
        }


    return {
        "pages": pages,
        "file_type": file_type
    }


# ------------------------------------------------------------
# 5. PAGE-AWARE CHUNKING
# ------------------------------------------------------------

def create_document_chunks(
    pages,
    chunk_size=100,
    overlap=20
):

    chunks = []

    chunk_id = 1

    for page_data in pages:

        page_number = page_data["page"]

        words = page_data["text"].split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk_words = words[
                start:end
            ]

            chunk_text = " ".join(
                chunk_words
            )

            if chunk_text.strip():

                chunks.append({

                    "chunk_id":
                        chunk_id,

                    "page":
                        page_number,

                    "text":
                        chunk_text
                })

                chunk_id += 1


            if end >= len(words):
                break

            start += (
                chunk_size - overlap
            )

    return chunks


# ------------------------------------------------------------
# 6. DOCUMENT SEMANTIC SEARCH
# ------------------------------------------------------------

def search_uploaded_document(
    file_path,
    query,
    top_k=3
):

    if not query or not query.strip():

        return {
            "error":
                "Please enter a search query."
        }


    start_time = time.time()


    # --------------------------------------------------------
    # LOAD DOCUMENT
    # --------------------------------------------------------

    document = load_document(
        file_path
    )

    if "error" in document:

        return document


    pages = document["pages"]


    # --------------------------------------------------------
    # CREATE CHUNKS
    # --------------------------------------------------------

    chunks = create_document_chunks(
        pages
    )


    if not chunks:

        return {
            "error":
                "No searchable text was found."
        }


    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]


    # --------------------------------------------------------
    # CREATE DOCUMENT EMBEDDINGS
    # --------------------------------------------------------

    chunk_embeddings = (
        embedding_model.encode(
            chunk_texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
    )


    # --------------------------------------------------------
    # CREATE QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = (
        embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
    )


    # --------------------------------------------------------
    # COSINE SIMILARITY
    # --------------------------------------------------------

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]


    # --------------------------------------------------------
    # RANK RESULTS
    # --------------------------------------------------------

    top_k = min(
        int(top_k),
        len(chunks)
    )

    ranked_indices = np.argsort(
        similarities
    )[::-1][:top_k]


    results = []

    for rank, index in enumerate(
        ranked_indices,
        start=1
    ):

        chunk = chunks[index]

        results.append({

            "rank":
                rank,

            "page":
                chunk["page"],

            "chunk_id":
                chunk["chunk_id"],

            "similarity":
                round(
                    float(
                        similarities[index]
                    ),
                    4
                ),

            "text":
                chunk["text"]
        })


    elapsed = round(
        (time.time() - start_time)
        * 1000,
        2
    )


    return {

        "task":
            "Document Semantic Search",

        "file_type":
            document["file_type"],

        "pages":
            len(pages),

        "chunks":
            len(chunks),

        "results":
            results,

        "model":
            EMBEDDING_MODEL,

        "search_time_ms":
            elapsed
    }


# ------------------------------------------------------------
# 7. GRADIO FUNCTION
# ------------------------------------------------------------

def document_search_ui(
    file,
    query,
    top_k
):

    try:

        if file is None:

            return (
                "Please upload a PDF or TXT file.",
                ""
            )


        result = search_uploaded_document(
            file,
            query,
            top_k
        )


        if "error" in result:

            return (
                result["error"],
                ""
            )


        output = ""


        for item in result["results"]:

            output += (
                f"RANK {item['rank']}\n"
                f"Page: {item['page']}\n"
                f"Chunk: {item['chunk_id']}\n"
                f"Similarity: "
                f"{item['similarity']}\n\n"
                f"{item['text']}\n\n"
                f"{'=' * 60}\n\n"
            )


        info = f"""
File Type: {result['file_type']}

Pages Processed: {result['pages']}

Document Chunks: {result['chunks']}

Embedding Model:
{result['model']}

Search Time:
{result['search_time_ms']} ms
"""


        return (
            output,
            info
        )


    except Exception as e:

        return (
            f"Document processing error:\n{str(e)}",
            ""
        )


# ------------------------------------------------------------
# 8. GRADIO INTERFACE
# ------------------------------------------------------------

document_demo = gr.Interface(

    fn=document_search_ui,

    inputs=[

        gr.File(
            label="Upload Document",
            file_types=[
                ".pdf",
                ".txt"
            ],
            type="filepath"
        ),

        gr.Textbox(
            label="Search Query",
            placeholder=(
                "What information "
                "are you looking for?"
            )
        ),

        gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Top Results"
        )
    ],

    outputs=[

        gr.Textbox(
            lines=18,
            label="Relevant Passages"
        ),

        gr.Textbox(
            lines=8,
            label="Document Information"
        )
    ],

    title=(
        "📄 TransformerLab — "
        "Document Intelligence"
    ),

    description=(
        "Upload PDF or TXT documents and "
        "search them using Transformer "
        "embeddings and semantic similarity."
    )
)


# ------------------------------------------------------------
# 9. LAUNCH
# ------------------------------------------------------------

document_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e48c231a6a8144abf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [71]:
# ============================================================
# MODULE 10 — MODEL INSIGHTS + TOKENIZER INSPECTOR
# TransformerLab AI
# ============================================================

import time
import torch
import pandas as pd
import gradio as gr


# ------------------------------------------------------------
# 1. MODEL REGISTRY
# ------------------------------------------------------------

MODEL_REGISTRY = {

    "Sentiment — DistilBERT": {
        "pipeline": sentiment_model,
        "task": "Sentiment Analysis",
        "architecture": "Encoder-only Transformer"
    },

    "Zero-Shot — BART": {
        "pipeline": zero_shot_model,
        "task": "Zero-Shot Classification",
        "architecture": "Encoder-Decoder Transformer"
    },

    "NER — BERT": {
        "pipeline": ner_model,
        "task": "Named Entity Recognition",
        "architecture": "Encoder-only Transformer"
    },

    "Question Answering — DistilBERT": {
        "pipeline": qa_model,
        "task": "Extractive Question Answering",
        "architecture": "Encoder-only Transformer"
    },

    "Summarization — DistilBART": {
        "pipeline": summarization_model,
        "task": "Text Summarization",
        "architecture": "Encoder-Decoder Transformer"
    },

    "Text Generation — SmolLM2": {
        "pipeline": text_generator,
        "task": "Autoregressive Text Generation",
        "architecture": "Decoder-only Transformer"
    }
}


# ------------------------------------------------------------
# 2. PARAMETER COUNTER
# ------------------------------------------------------------

def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable


# ------------------------------------------------------------
# 3. TOKENIZER INSPECTOR
# ------------------------------------------------------------

def inspect_tokenizer(text, model_choice):

    if not text or not text.strip():

        return (
            pd.DataFrame(),
            "Please enter some text."
        )


    selected = MODEL_REGISTRY[
        model_choice
    ]

    pipe = selected["pipeline"]

    tokenizer = pipe.tokenizer

    model = pipe.model


    # --------------------------------------------------------
    # TOKENIZATION
    # --------------------------------------------------------

    encoded = tokenizer(
        text,
        add_special_tokens=True
    )

    token_ids = encoded[
        "input_ids"
    ]

    tokens = tokenizer.convert_ids_to_tokens(
        token_ids
    )


    # --------------------------------------------------------
    # BUILD TOKEN TABLE
    # --------------------------------------------------------

    token_data = []

    for position, (
        token,
        token_id
    ) in enumerate(
        zip(tokens, token_ids)
    ):

        token_data.append({

            "Position":
                position,

            "Token":
                token,

            "Token ID":
                token_id
        })


    token_df = pd.DataFrame(
        token_data
    )


    # --------------------------------------------------------
    # MODEL INFORMATION
    # --------------------------------------------------------

    total_params, trainable_params = (
        count_parameters(model)
    )


    config = model.config


    hidden_size = getattr(
        config,
        "hidden_size",
        "N/A"
    )

    if hidden_size == "N/A":

        hidden_size = getattr(
            config,
            "d_model",
            "N/A"
        )


    num_layers = getattr(
        config,
        "num_hidden_layers",
        "N/A"
    )

    if num_layers == "N/A":

        num_layers = getattr(
            config,
            "encoder_layers",
            "N/A"
        )


    attention_heads = getattr(
        config,
        "num_attention_heads",
        "N/A"
    )

    if attention_heads == "N/A":

        attention_heads = getattr(
            config,
            "encoder_attention_heads",
            "N/A"
        )


    vocab_size = getattr(
        config,
        "vocab_size",
        len(tokenizer)
    )


    info = f"""
TASK
{selected['task']}

ARCHITECTURE
{selected['architecture']}

MODEL
{getattr(config, '_name_or_path', model_choice)}

TOKENIZER
{tokenizer.__class__.__name__}

TOKENS IN INPUT
{len(token_ids)}

VOCABULARY SIZE
{vocab_size:,}

HIDDEN DIMENSION
{hidden_size}

TRANSFORMER LAYERS
{num_layers}

ATTENTION HEADS
{attention_heads}

TOTAL PARAMETERS
{total_params:,}

TRAINABLE PARAMETERS
{trainable_params:,}
"""


    return (
        token_df,
        info
    )


# ------------------------------------------------------------
# 4. TOKEN COUNT FUNCTION
# ------------------------------------------------------------

def tokenizer_stats(text, model_choice):

    selected = MODEL_REGISTRY[
        model_choice
    ]

    tokenizer = selected[
        "pipeline"
    ].tokenizer


    if not text:

        return (
            0,
            0,
            0
        )


    words = len(
        text.split()
    )


    tokens = tokenizer.encode(
        text,
        add_special_tokens=True
    )


    return (
        len(text),
        words,
        len(tokens)
    )


# ------------------------------------------------------------
# 5. GRADIO INTERFACE
# ------------------------------------------------------------

with gr.Blocks(
    title="TransformerLab Model Insights"
) as insights_demo:


    gr.Markdown(
        """
        # 🧠 TransformerLab — Model Insights

        Inspect how Transformer models tokenize text
        and explore their architecture.
        """
    )


    with gr.Row():

        model_choice = gr.Dropdown(

            choices=list(
                MODEL_REGISTRY.keys()
            ),

            value="Sentiment — DistilBERT",

            label="Select Transformer Model"
        )


    input_text = gr.Textbox(

        lines=5,

        label="Input Text",

        placeholder=(
            "Enter text to inspect "
            "its tokenization..."
        )
    )


    inspect_button = gr.Button(
        "Inspect Model"
    )


    with gr.Row():

        char_count = gr.Number(
            label="Characters"
        )

        word_count = gr.Number(
            label="Words"
        )

        token_count = gr.Number(
            label="Tokens"
        )


    token_table = gr.Dataframe(

        headers=[
            "Position",
            "Token",
            "Token ID"
        ],

        label="Tokenizer Output",

        interactive=False
    )


    model_information = gr.Textbox(

        lines=18,

        label="Model Architecture"
    )


    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    inspect_button.click(

        fn=inspect_tokenizer,

        inputs=[
            input_text,
            model_choice
        ],

        outputs=[
            token_table,
            model_information
        ]
    )


    input_text.change(

        fn=tokenizer_stats,

        inputs=[
            input_text,
            model_choice
        ],

        outputs=[
            char_count,
            word_count,
            token_count
        ]
    )


    model_choice.change(

        fn=tokenizer_stats,

        inputs=[
            input_text,
            model_choice
        ],

        outputs=[
            char_count,
            word_count,
            token_count
        ]
    )


# ------------------------------------------------------------
# 6. LAUNCH
# ------------------------------------------------------------

insights_demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4f34d88d72932bb858.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [72]:
# ============================================================
# MODULE 11 — TRANSFORMER MODEL COMPARISON
# TransformerLab AI
# ============================================================

import time
import torch
import pandas as pd
import gradio as gr
from transformers import pipeline


# ------------------------------------------------------------
# 1. MODELS
# ------------------------------------------------------------

MODEL_A_NAME = (
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

MODEL_B_NAME = (
    "textattack/bert-base-uncased-SST-2"
)


# Reuse our existing DistilBERT sentiment model
comparison_model_a = sentiment_model


# Load BERT comparison model
comparison_model_b = pipeline(
    "sentiment-analysis",
    model=MODEL_B_NAME,
    device=device
)


# ------------------------------------------------------------
# 2. PARAMETER COUNTER
# ------------------------------------------------------------

def get_parameter_count(model):

    return sum(
        parameter.numel()
        for parameter in model.parameters()
    )


# ------------------------------------------------------------
# 3. APPROXIMATE MODEL MEMORY
# ------------------------------------------------------------

def get_model_memory_mb(model):

    total_bytes = 0

    for parameter in model.parameters():

        total_bytes += (
            parameter.numel()
            * parameter.element_size()
        )

    for buffer in model.buffers():

        total_bytes += (
            buffer.numel()
            * buffer.element_size()
        )

    return round(
        total_bytes / (1024 ** 2),
        2
    )


# ------------------------------------------------------------
# 4. SINGLE MODEL BENCHMARK
# ------------------------------------------------------------

def benchmark_sentiment_model(
    model_pipeline,
    text
):

    # GPU synchronization gives more accurate timing
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()


    result = model_pipeline(
        text
    )[0]


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    latency = (
        time.perf_counter() - start
    ) * 1000


    return {

        "label":
            result["label"],

        "confidence":
            round(
                float(result["score"]) * 100,
                2
            ),

        "latency_ms":
            round(latency, 2),

        "parameters":
            get_parameter_count(
                model_pipeline.model
            ),

        "memory_mb":
            get_model_memory_mb(
                model_pipeline.model
            )
    }


# ------------------------------------------------------------
# 5. MODEL COMPARISON
# ------------------------------------------------------------

def compare_models(text):

    if not text or not text.strip():

        return (
            pd.DataFrame(),
            "Please enter some text."
        )


    # MODEL A
    result_a = benchmark_sentiment_model(
        comparison_model_a,
        text
    )


    # MODEL B
    result_b = benchmark_sentiment_model(
        comparison_model_b,
        text
    )


    # --------------------------------------------------------
    # CREATE COMPARISON TABLE
    # --------------------------------------------------------

    comparison_data = [

        {
            "Model":
                "DistilBERT SST-2",

            "Prediction":
                result_a["label"],

            "Confidence (%)":
                result_a["confidence"],

            "Latency (ms)":
                result_a["latency_ms"],

            "Parameters":
                f"{result_a['parameters']:,}",

            "Approx Memory (MB)":
                result_a["memory_mb"]
        },

        {
            "Model":
                "BERT SST-2",

            "Prediction":
                result_b["label"],

            "Confidence (%)":
                result_b["confidence"],

            "Latency (ms)":
                result_b["latency_ms"],

            "Parameters":
                f"{result_b['parameters']:,}",

            "Approx Memory (MB)":
                result_b["memory_mb"]
        }
    ]


    comparison_df = pd.DataFrame(
        comparison_data
    )


    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    if (
        result_a["latency_ms"]
        <
        result_b["latency_ms"]
    ):

        faster_model = "DistilBERT"

    else:

        faster_model = "BERT"


    if (
        result_a["confidence"]
        >
        result_b["confidence"]
    ):

        confidence_model = "DistilBERT"

    elif (
        result_b["confidence"]
        >
        result_a["confidence"]
    ):

        confidence_model = "BERT"

    else:

        confidence_model = "Tie"


    agreement = (
        result_a["label"]
        ==
        result_b["label"]
    )


    summary = f"""
MODEL COMPARISON SUMMARY

Prediction Agreement:
{"Yes" if agreement else "No"}

Faster Model:
{faster_model}

Higher Confidence:
{confidence_model}

DistilBERT Latency:
{result_a['latency_ms']} ms

BERT Latency:
{result_b['latency_ms']} ms

DistilBERT Parameters:
{result_a['parameters']:,}

BERT Parameters:
{result_b['parameters']:,}

NOTE:
Latency depends on the current hardware and runtime.
Confidence does not directly mean that one model is
more accurate overall.
"""


    return (
        comparison_df,
        summary
    )


# ------------------------------------------------------------
# 6. GRADIO INTERFACE
# ------------------------------------------------------------

with gr.Blocks(
    title="TransformerLab Model Comparison"
) as comparison_demo:


    gr.Markdown(
        """
        # ⚖️ TransformerLab — Model Comparison

        Compare two Transformer models performing
        the same NLP task on identical input.
        """
    )


    comparison_input = gr.Textbox(

        lines=6,

        label="Input Text",

        placeholder=(
            "Enter text for sentiment analysis..."
        )
    )


    compare_button = gr.Button(
        "Compare Models"
    )


    comparison_table = gr.Dataframe(

        headers=[
            "Model",
            "Prediction",
            "Confidence (%)",
            "Latency (ms)",
            "Parameters",
            "Approx Memory (MB)"
        ],

        label="Model Benchmark",

        interactive=False
    )


    comparison_summary = gr.Textbox(

        lines=18,

        label="Comparison Summary"
    )


    compare_button.click(

        fn=compare_models,

        inputs=[
            comparison_input
        ],

        outputs=[
            comparison_table,
            comparison_summary
        ]
    )


# ------------------------------------------------------------
# 7. LAUNCH
# ------------------------------------------------------------

comparison_demo.launch()

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1f553d2ae4c5da435.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [74]:
# ============================================================
# TRANSFORMERLAB AI
# FINAL UNIFIED GRADIO DASHBOARD
# ============================================================

import gradio as gr
import pandas as pd


# ============================================================
# UI WRAPPER FUNCTIONS
# ============================================================


# ------------------------------------------------------------
# SENTIMENT
# ------------------------------------------------------------

def final_sentiment(text):

    if not text or not text.strip():
        return "Please enter some text."

    result = analyze_sentiment(text)

    if "error" in result:
        return result["error"]

    return f"""
Prediction: {result['label']}

Confidence: {result['confidence']}%

Model: {result['model']}
"""


# ------------------------------------------------------------
# ZERO-SHOT
# ------------------------------------------------------------

def final_zero_shot(text, labels):

    if not text or not text.strip():
        return "Please enter some text."

    label_list = [
        label.strip()
        for label in labels.split(",")
        if label.strip()
    ]

    if not label_list:
        return "Please provide candidate labels."

    result = classify_zero_shot(
        text,
        label_list
    )

    if "error" in result:
        return result["error"]

    output = ""

    for prediction in result["predictions"]:

        output += (
            f"{prediction['label']}: "
            f"{prediction['confidence']}%\n"
        )

    return output


# ------------------------------------------------------------
# NER
# ------------------------------------------------------------

def final_ner(text):

    result = analyze_entities(text)

    if "error" in result:
        return result["error"]

    if not result["entities"]:
        return "No named entities detected."

    output = ""

    for entity in result["entities"]:

        output += (
            f"{entity['text']} → "
            f"{entity['type']} "
            f"({entity['confidence']}%)\n"
        )

    return output


# ------------------------------------------------------------
# QUESTION ANSWERING
# ------------------------------------------------------------

def final_qa(context, question):

    result = answer_question(
        question,
        context
    )

    if "error" in result:
        return result["error"], ""

    answer = f"""
Answer: {result['answer']}

Confidence: {result['confidence']}%

Model: {result['model']}
"""

    evidence = highlight_answer(
        context,
        result["start"],
        result["end"]
    )

    return answer, evidence


# ------------------------------------------------------------
# SUMMARIZATION
# ------------------------------------------------------------

def final_summary(text, length):

    result = summarize_text(
        text,
        length
    )

    if "error" in result:
        return result["error"], ""

    statistics = f"""
Original Words: {result['original_words']}

Summary Words: {result['summary_words']}

Compression: {result['compression']}%

Model: {result['model']}
"""

    return (
        result["summary"],
        statistics
    )


# ------------------------------------------------------------
# TRANSLATION
# ------------------------------------------------------------

def final_translation(
    text,
    language_pair
):

    result = translate_text(
        text,
        language_pair
    )

    if "error" in result:
        return result["error"], ""

    info = f"""
Language Pair: {result['language_pair']}

Model: {result['model']}

Input Tokens: {result['input_tokens']}

Output Tokens: {result['output_tokens']}
"""

    return (
        result["translation"],
        info
    )


# ------------------------------------------------------------
# TEXT GENERATION
# ------------------------------------------------------------

def final_generation(
    prompt,
    max_tokens,
    temperature,
    top_p,
    sequences
):

    result = generate_text(
        prompt,
        max_tokens,
        temperature,
        top_p,
        sequences
    )

    if "error" in result:
        return result["error"], ""

    output = ""

    for generation in result[
        "generations"
    ]:

        output += (
            f"GENERATION "
            f"{generation['sequence']}\n\n"
            f"{generation['text']}\n\n"
            f"{'-' * 50}\n\n"
        )

    info = f"""
Model: {result['model']}

Temperature: {result['temperature']}

Top-p: {result['top_p']}

Input Tokens: {result['input_tokens']}

Inference Time: {result['inference_time_ms']} ms
"""

    return output, info


# ============================================================
# FINAL APPLICATION
# ============================================================

with gr.Blocks(
    title="TransformerLab AI"
) as app:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
# ⚡ TransformerLab AI

### Interactive Transformer & NLP Intelligence Workbench

Explore Transformer models across classification,
information extraction, generation, semantic retrieval,
document intelligence and model analysis.

**Powered by Hugging Face Transformers,
Sentence Transformers & Gradio**
        """
    )


    # ========================================================
    # MAIN TABS
    # ========================================================

    with gr.Tabs():


        # ====================================================
        # TAB 1 — NLP WORKBENCH
        # ====================================================

        with gr.Tab(
            "🧠 NLP Workbench"
        ):

            gr.Markdown(
                """
## NLP Understanding

Analyze text using Transformer-based
language understanding models.
                """
            )


            # -----------------------------------------------
            # SENTIMENT
            # -----------------------------------------------

            with gr.Tab(
                "😊 Sentiment"
            ):

                sentiment_input = (
                    gr.Textbox(
                        lines=6,
                        label="Text",
                        placeholder=(
                            "Enter text for "
                            "sentiment analysis..."
                        )
                    )
                )

                sentiment_button = (
                    gr.Button(
                        "Analyze Sentiment",
                        variant="primary"
                    )
                )

                sentiment_output = (
                    gr.Textbox(
                        lines=6,
                        label="Result"
                    )
                )

                sentiment_button.click(
                    final_sentiment,
                    sentiment_input,
                    sentiment_output
                )


            # -----------------------------------------------
            # ZERO SHOT
            # -----------------------------------------------

            with gr.Tab(
                "🏷️ Zero-Shot"
            ):

                zero_text = gr.Textbox(
                    lines=6,
                    label="Text"
                )

                zero_labels = gr.Textbox(
                    label="Candidate Labels",
                    placeholder=(
                        "technology, finance, "
                        "sports, healthcare"
                    )
                )

                zero_button = gr.Button(
                    "Classify",
                    variant="primary"
                )

                zero_output = gr.Textbox(
                    lines=8,
                    label="Ranked Categories"
                )

                zero_button.click(
                    final_zero_shot,
                    [
                        zero_text,
                        zero_labels
                    ],
                    zero_output
                )


            # -----------------------------------------------
            # NER
            # -----------------------------------------------

            with gr.Tab(
                "👤 Named Entities"
            ):

                ner_input = gr.Textbox(
                    lines=8,
                    label="Text"
                )

                ner_button = gr.Button(
                    "Detect Entities",
                    variant="primary"
                )

                ner_output = gr.Textbox(
                    lines=10,
                    label="Entities"
                )

                ner_button.click(
                    final_ner,
                    ner_input,
                    ner_output
                )


            # -----------------------------------------------
            # QUESTION ANSWERING
            # -----------------------------------------------

            with gr.Tab(
                "❓ Question Answering"
            ):

                qa_context = gr.Textbox(
                    lines=10,
                    label="Context"
                )

                qa_question = gr.Textbox(
                    label="Question"
                )

                qa_button = gr.Button(
                    "Find Answer",
                    variant="primary"
                )

                qa_answer = gr.Textbox(
                    lines=6,
                    label="Answer"
                )

                qa_evidence = gr.Markdown(
                    label="Source Evidence"
                )

                qa_button.click(
                    final_qa,
                    [
                        qa_context,
                        qa_question
                    ],
                    [
                        qa_answer,
                        qa_evidence
                    ]
                )


        # ====================================================
        # TAB 2 — GENERATIVE NLP
        # ====================================================

        with gr.Tab(
            "✨ Generative NLP"
        ):

            gr.Markdown(
                """
## Generative Transformers

Explore sequence-to-sequence and
autoregressive Transformer models.
                """
            )


            # -----------------------------------------------
            # SUMMARIZATION
            # -----------------------------------------------

            with gr.Tab(
                "📝 Summarization"
            ):

                summary_input = gr.Textbox(
                    lines=12,
                    label="Document"
                )

                summary_length = gr.Radio(
                    [
                        "Short",
                        "Medium",
                        "Detailed"
                    ],
                    value="Medium",
                    label="Summary Length"
                )

                summary_button = gr.Button(
                    "Generate Summary",
                    variant="primary"
                )

                summary_output = gr.Textbox(
                    lines=10,
                    label="Summary"
                )

                summary_stats = gr.Textbox(
                    lines=6,
                    label="Statistics"
                )

                summary_button.click(
                    final_summary,
                    [
                        summary_input,
                        summary_length
                    ],
                    [
                        summary_output,
                        summary_stats
                    ]
                )


            # -----------------------------------------------
            # TRANSLATION
            # -----------------------------------------------

            with gr.Tab(
                "🌍 Translation"
            ):

                translation_input = (
                    gr.Textbox(
                        lines=8,
                        label="Text"
                    )
                )

                translation_language = (
                    gr.Dropdown(
                        choices=list(
                            TRANSLATION_MODELS.keys()
                        ),
                        value=(
                            "English → French"
                        ),
                        label="Language Pair"
                    )
                )

                translation_button = (
                    gr.Button(
                        "Translate",
                        variant="primary"
                    )
                )

                translation_output = (
                    gr.Textbox(
                        lines=8,
                        label="Translation"
                    )
                )

                translation_info = (
                    gr.Textbox(
                        lines=6,
                        label="Model Information"
                    )
                )

                translation_button.click(
                    final_translation,
                    [
                        translation_input,
                        translation_language
                    ],
                    [
                        translation_output,
                        translation_info
                    ]
                )


            # -----------------------------------------------
            # TEXT GENERATION
            # -----------------------------------------------

            with gr.Tab(
                "✍️ Text Generation"
            ):

                generation_prompt = (
                    gr.Textbox(
                        lines=6,
                        label="Prompt"
                    )
                )

                with gr.Row():

                    generation_tokens = (
                        gr.Slider(
                            20,
                            250,
                            value=100,
                            step=10,
                            label="Max Tokens"
                        )
                    )

                    generation_temperature = (
                        gr.Slider(
                            0.1,
                            1.5,
                            value=0.7,
                            step=0.1,
                            label="Temperature"
                        )
                    )

                with gr.Row():

                    generation_top_p = (
                        gr.Slider(
                            0.1,
                            1.0,
                            value=0.9,
                            step=0.05,
                            label="Top-p"
                        )
                    )

                    generation_sequences = (
                        gr.Slider(
                            1,
                            3,
                            value=1,
                            step=1,
                            label="Sequences"
                        )
                    )

                generation_button = (
                    gr.Button(
                        "Generate",
                        variant="primary"
                    )
                )

                generation_output = (
                    gr.Textbox(
                        lines=15,
                        label="Generated Text"
                    )
                )

                generation_info = (
                    gr.Textbox(
                        lines=7,
                        label="Generation Info"
                    )
                )

                generation_button.click(
                    final_generation,
                    [
                        generation_prompt,
                        generation_tokens,
                        generation_temperature,
                        generation_top_p,
                        generation_sequences
                    ],
                    [
                        generation_output,
                        generation_info
                    ]
                )


        # ====================================================
        # TAB 3 — SEMANTIC SEARCH
        # ====================================================

        with gr.Tab(
            "🔎 Semantic Search"
        ):

            gr.Markdown(
                """
## Search by Meaning

Transform text into dense embeddings
and retrieve passages using cosine similarity.
                """
            )

            semantic_document = gr.Textbox(
                lines=15,
                label="Document"
            )

            semantic_query = gr.Textbox(
                label="Search Query"
            )

            semantic_top_k = gr.Slider(
                1,
                5,
                value=3,
                step=1,
                label="Top Results"
            )

            semantic_button = gr.Button(
                "Search",
                variant="primary"
            )

            semantic_results = gr.Textbox(
                lines=16,
                label="Ranked Results"
            )

            semantic_info = gr.Textbox(
                lines=6,
                label="Embedding Information"
            )

            semantic_button.click(
                semantic_search_ui,
                [
                    semantic_document,
                    semantic_query,
                    semantic_top_k
                ],
                [
                    semantic_results,
                    semantic_info
                ]
            )


        # ====================================================
        # TAB 4 — DOCUMENT INTELLIGENCE
        # ====================================================

        with gr.Tab(
            "📄 Document Intelligence"
        ):

            gr.Markdown(
                """
## PDF / TXT Semantic Search

Upload a document and retrieve
relevant passages using Transformer embeddings.
                """
            )

            document_file = gr.File(
                label="Upload PDF / TXT",
                file_types=[
                    ".pdf",
                    ".txt"
                ],
                type="filepath"
            )

            document_query = gr.Textbox(
                label="Question / Search Query"
            )

            document_top_k = gr.Slider(
                1,
                5,
                value=3,
                step=1,
                label="Top Results"
            )

            document_button = gr.Button(
                "Search Document",
                variant="primary"
            )

            document_results = gr.Textbox(
                lines=18,
                label="Relevant Passages"
            )

            document_info = gr.Textbox(
                lines=8,
                label="Document Information"
            )

            document_button.click(
                document_search_ui,
                [
                    document_file,
                    document_query,
                    document_top_k
                ],
                [
                    document_results,
                    document_info
                ]
            )


        # ====================================================
        # TAB 5 — MODEL LAB
        # ====================================================

        with gr.Tab(
            "🧪 Model Lab"
        ):


            # -----------------------------------------------
            # TOKENIZER / MODEL INSIGHTS
            # -----------------------------------------------

            with gr.Tab(
                "🔬 Model Insights"
            ):

                insight_model = gr.Dropdown(
                    choices=list(
                        MODEL_REGISTRY.keys()
                    ),
                    value=(
                        "Sentiment — DistilBERT"
                    ),
                    label="Transformer Model"
                )

                insight_text = gr.Textbox(
                    lines=6,
                    label="Text to Inspect"
                )

                insight_button = gr.Button(
                    "Inspect",
                    variant="primary"
                )

                with gr.Row():

                    insight_chars = gr.Number(
                        label="Characters"
                    )

                    insight_words = gr.Number(
                        label="Words"
                    )

                    insight_tokens = gr.Number(
                        label="Tokens"
                    )

                insight_table = gr.Dataframe(
                    label="Tokenization",
                    interactive=False
                )

                insight_info = gr.Textbox(
                    lines=18,
                    label="Architecture"
                )

                insight_button.click(
                    inspect_tokenizer,
                    [
                        insight_text,
                        insight_model
                    ],
                    [
                        insight_table,
                        insight_info
                    ]
                )

                insight_text.change(
                    tokenizer_stats,
                    [
                        insight_text,
                        insight_model
                    ],
                    [
                        insight_chars,
                        insight_words,
                        insight_tokens
                    ]
                )


            # -----------------------------------------------
            # MODEL COMPARISON
            # -----------------------------------------------

            with gr.Tab(
                "⚖️ Model Comparison"
            ):

                compare_input = gr.Textbox(
                    lines=6,
                    label="Sentiment Text"
                )

                compare_button = gr.Button(
                    "Compare Models",
                    variant="primary"
                )

                compare_table = gr.Dataframe(
                    label="Benchmark Results",
                    interactive=False
                )

                compare_summary = gr.Textbox(
                    lines=18,
                    label="Comparison"
                )

                compare_button.click(
                    compare_models,
                    compare_input,
                    [
                        compare_table,
                        compare_summary
                    ]
                )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.Markdown(
        """
---

### TransformerLab AI

**Tasks:** Classification • NER • QA • Summarization •
Translation • Generation • Semantic Retrieval

**Model Families:** BERT • DistilBERT • BART • MarianMT •
SmolLM2 • Sentence Transformers
        """
    )


# ============================================================
# LAUNCH FINAL APPLICATION
# ============================================================

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://029c1eb6cd1c47e2d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
